In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# ジョブの監視またはキャンセル

ワークロードの一覧は、[ワークロードページ](https://quantum.cloud.ibm.com/workloads)で確認できます。

## ジョブのステータスを確認する
[ワークロードテーブル](https://quantum.cloud.ibm.com/workloads)を開き、「Status」列でジョブが完了したか失敗したかを確認してください。

## 残り使用量を確認する
[インスタンステーブル](https://quantum.cloud.ibm.com/instances)を開き、残り使用量を確認したいプランに対応するタブを選択してください。プランの合計使用時間と残り時間が表示されます。

## 送信されたジョブおよびワークロード数のメトリクスを確認する
[アナリティクスページ](https://quantum.cloud.ibm.com/analytics)にアクセスすると、送信されたジョブの総数、バッチワークロードとセッションワークロードの件数を確認できます。アナリティクスページは、ご自身が所有または管理しているアカウントのみ表示可能です。

## ジョブを監視する
ジョブインスタンスを使用して、適切なコマンドを呼び出すことでジョブのステータスを確認したり、結果を取得したりできます。

|                               |                                                                                                                                                                                                              |
| ----------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| job.result()                  | ジョブ完了直後に結果を確認します。ジョブの結果はジョブ完了後に利用可能になります。そのため、`job.result()` はジョブが完了するまでブロッキング呼び出しとなります。                               |
| job.job\_id()                 | そのジョブを一意に識別するIDを返します。後でジョブの結果を取得するにはジョブIDが必要です。そのため、後で取得する可能性のあるジョブのIDは保存しておくことをお勧めします。 |
| job.status()                  | ジョブのステータスを確認します。                                                                                                                                                                                        |
| job = service.job(\<job\_id>) | 以前に送信したジョブを取得します。この呼び出しにはジョブIDが必要です。                                                                                                                                      |

<span id="retrieve-later"></span>
## 後でジョブ結果を取得する
`service.job(\<job\_id>)` を呼び出すことで、以前に送信したジョブを取得できます。ジョブIDが手元にない場合、または複数のジョブを一度に取得したい場合（廃止されたQPU（量子処理ユニット）のジョブを含む）は、代わりにオプションフィルターを指定して `service.jobs()` を呼び出してください。詳細は [QiskitRuntimeService.jobs](../api/qiskit-ibm-runtime/qiskit-runtime-service#jobs) を参照してください。

> **Note:** `service.jobs()` は、非推奨の `qiskit-ibm-provider` パッケージで実行されたジョブも返します。旧来の（同様に非推奨の）`qiskit-ibmq-provider` パッケージで送信されたジョブは、もはや利用できません。

### 例
この例では、`my_backend` で実行された最新の10件のランタイムジョブを返します。

In [ ]:
# This cell is hidden from users
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
import numpy as np


my_backend = "ibm_torino"
service = QiskitRuntimeService()
# backend = service.backend(my_backend)
backend = service.least_busy()

# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)

params = np.random.uniform(size=(2, 3)).T

sampler_pub = (transpiled_circuit, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.
sampler = SamplerV2(mode=backend)
job = sampler.run([sampler_pub], shots=4)
print(job.job_id())

d305ck0ocacs73ajagvg


In [2]:
result = job.result()


spans = job.result().metadata["execution"]["execution_spans"]
print(spans)

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])


In [3]:
params = np.random.uniform(size=(2, 3))
params

array([[0.2260416 , 0.8747859 , 0.44361995],
       [0.94700856, 0.96826017, 0.98426562]])

In [4]:
mask = spans[0].mask(0)
mask

array([[[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]]])

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Initialize the account first.
service = QiskitRuntimeService()
# Use `limit` to retrieve a specific number of jobs. The default `limit` is 10.
service.jobs(backend_name=my_backend)

## Cancel a job

You can cancel a job from the IBM Quantum Platform dashboard either on the Workloads page or the details page for a specific workload. On the Workloads page, click the overflow menu at the end of the row for that workload, and select Cancel. If you are on the details page for a specific workload, use the Actions dropdown at the top of the page, and select Cancel.

In Qiskit, use `job.cancel()` to cancel a job.

## Next steps

<Admonition type="tip" title="Recommendations">
    - Try the [Grover's algorithm](/docs/tutorials/grovers-algorithm) tutorial.
    - Learn more about [Sampler execution spans](/docs/guides/sampler-input-output#execution-spans)
</Admonition>